# Phase 3 Post-RL Evaluation — Airline Multi-Step Tasks

This notebook evaluates a HuggingFace fine-tuned model against the `airline_multistep`
test split and prints a **Before / After** comparison table.

## How it works

1. **Install** `vllm` (OpenAI-compatible local server) + `tau2` from the `airline-tasks` branch.
2. **Set API keys** for Groq (Before baseline) and optionally Anthropic/OpenAI.
3. **Serve the tuned model** locally via `vllm` on port 8000.
4. **Run Before baseline** on the test split using the original Groq model.
5. **Run After eval** on the test split with the tuned model via vllm.
6. **Compare** rewards side-by-side.

> **Runtime**: Use a **T4 GPU** (free in Colab) — `Runtime → Change runtime type → T4 GPU`.
> The tuned model (`Qwen2.5-0.5B-Instruct`) needs ~1 GB VRAM; a T4 has 16 GB.

## Cell 1 — Install dependencies

In [ ]:
# Install vllm (OpenAI-compatible server) and tau2 from the airline-tasks branch.
# This takes ~3 minutes on Colab.
!pip install -q vllm
!pip install -q git+https://github.com/sierra-research/tau2-bench.git@airline-tasks

# Verify tau2 CLI is available
!tau2 --help | head -5

## Cell 2 — Set API keys

In [ ]:
import os

# ── Required ────────────────────────────────────────────────────────────────────
# Groq: free tier, used for the Before baseline and user simulator.
# Get a key at https://console.groq.com
os.environ["GROQ_API_KEY"] = "gsk_..."  # <-- replace with your Groq key

# ── Optional ────────────────────────────────────────────────────────────────────
# Uncomment if you want Claude or GPT as the user simulator instead of Groq.
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"]    = "sk-..."

# HuggingFace token — only needed if the tuned model repo is private.
# os.environ["HF_TOKEN"] = "hf_..."

print("Keys set.")

## Cell 3 — Start vllm server (serves the tuned model on port 8000)

vllm exposes an OpenAI-compatible REST API.  
We run it in the background and wait for it to be ready before making requests.

> **Change `HF_MODEL_ID`** to point to your own tuned model repo, or keep the
> default `ZahedRCV/airline-rl-tuned` if you pushed with `--push-to-hub`.

In [ ]:
import subprocess, time, requests

# ── Config ───────────────────────────────────────────────────────────────────────
HF_MODEL_ID = "ZahedRCV/airline-rl-tuned"   # <-- change to your HF repo if needed
VLLM_PORT   = 8000

# ── Start the server in a background process ────────────────────────────────────
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", HF_MODEL_ID,
        "--port", str(VLLM_PORT),
        "--dtype", "float16",          # T4 doesn't support bfloat16
        "--max-model-len", "4096",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# ── Wait until the server responds ─────────────────────────────────────────────
print("Waiting for vllm server to start ", end="", flush=True)
for _ in range(120):           # up to 2 minutes
    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            print(" ready!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(1)
else:
    raise RuntimeError("vllm server did not start within 2 minutes. Check logs above.")

# ── Show loaded model name ───────────────────────────────────────────────────────
models = requests.get(f"http://localhost:{VLLM_PORT}/v1/models").json()
print("Loaded model:", models["data"][0]["id"])

## Cell 4 — Before baseline (test split, Groq model)

Runs tau2 on the `test` split with the original Groq model to capture the
**Before** rewards. Results are saved to `baseline_test.json`.

In [ ]:
import subprocess

GROQ_MODEL = "groq/llama-3.3-70b-versatile"

result = subprocess.run(
    [
        "tau2", "run",
        "--domain",          "airline",
        "--task-set-name",   "airline_multistep",
        "--task-split-name", "test",
        "--agent-llm",       GROQ_MODEL,
        "--user-llm",        GROQ_MODEL,
        "--save-to",         "baseline_test",
    ],
    check=False,
)
print("Exit code:", result.returncode)

## Cell 5 — After evaluation (test split, tuned model via vllm)

Points tau2 at the local vllm server via `OPENAI_API_BASE`.  
litellm routes any `openai/<model-id>` request to that base URL.

In [ ]:
import os, subprocess, requests

VLLM_PORT   = 8000
HF_MODEL_ID = "ZahedRCV/airline-rl-tuned"   # must match Cell 3
GROQ_MODEL  = "groq/llama-3.3-70b-versatile"

# Discover model id from the vllm server (handles HF repo name → local id mapping)
models_resp = requests.get(f"http://localhost:{VLLM_PORT}/v1/models").json()
served_model_id = models_resp["data"][0]["id"]
AGENT_MODEL = f"openai/{served_model_id}"
print("Agent model:", AGENT_MODEL)

# Tell litellm to route openai/* to the local vllm server
env = {**os.environ, "OPENAI_API_BASE": f"http://localhost:{VLLM_PORT}/v1"}

result = subprocess.run(
    [
        "tau2", "run",
        "--domain",          "airline",
        "--task-set-name",   "airline_multistep",
        "--task-split-name", "test",
        "--agent-llm",       AGENT_MODEL,
        "--user-llm",        GROQ_MODEL,
        "--save-to",         "after_rl_test",
    ],
    env=env,
    check=False,
)
print("Exit code:", result.returncode)

## Cell 6 — Before / After comparison table

In [ ]:
import json
from pathlib import Path

def load_rewards(path: str | Path) -> dict[str, float]:
    """Return {task_id: reward} from a tau2 simulation JSON."""
    p = Path(path)
    # tau2 saves to data/simulations/<stem>.json
    candidates = [
        p,
        Path("data/simulations") / p.name,
        Path("data/simulations") / (p.stem + ".json"),
    ]
    found = next((c for c in candidates if c.exists()), None)
    if found is None:
        raise FileNotFoundError(f"Cannot find {path}. Checked: {candidates}")
    data = json.loads(found.read_text())
    # Handle both list-of-records and {results: [...]} formats
    records = data if isinstance(data, list) else data.get("results", [])
    return {r["task_id"]: r["reward"] for r in records}

before = load_rewards("baseline_test.json")
after  = load_rewards("after_rl_test.json")

all_ids = sorted(set(before) | set(after))

print("=" * 55)
print(f"{'Task':<14} {'Before':>12}    {'After':>12}")
print("-" * 55)
before_vals, after_vals = [], []
for tid in all_ids:
    b = before.get(tid)
    a = after.get(tid)
    b_str = f"{b:.4f}" if b is not None else "N/A"
    a_str = f"{a:.4f}" if a is not None else "N/A"
    if b is not None and a is not None:
        arrow = "↑" if a > b else ("↓" if a < b else "→")
        before_vals.append(b)
        after_vals.append(a)
    else:
        arrow = " "
    print(f"{tid:<14} {b_str:>12} {arrow:>4} {a_str:>12}")
print("=" * 55)
if before_vals and after_vals:
    avg_b = sum(before_vals) / len(before_vals)
    avg_a = sum(after_vals) / len(after_vals)
    arrow = "↑" if avg_a > avg_b else ("↓" if avg_a < avg_b else "→")
    print(f"{'AVERAGE':<14} {avg_b:>12.4f} {arrow:>4} {avg_a:>12.4f}")

## (Optional) Cell 7 — Stop the vllm server

In [ ]:
# Terminate the background vllm process when you're done.
try:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=10)
    print("vllm server stopped.")
except Exception as e:
    print(f"Could not stop vllm: {e}")

---

## Notes

### What `OPENAI_API_BASE` does
litellm routes any `openai/<model>` request to `OPENAI_API_BASE/v1/chat/completions`.  
This lets us point tau2 at the local vllm server without modifying any tau2 source.

### Running with a different base model
If you trained with `Qwen2.5-7B-Instruct` (recommended for real results), change
`HF_MODEL_ID` in Cells 3 and 5 to your repo. The rest of the notebook is identical.

### Training the model yourself
Run the full RL pipeline on your local machine or a Colab instance with the training script:
```bash
pip install transformers peft accelerate torch bitsandbytes
export GROQ_API_KEY=<key>
python -m tau2.scripts.rl_airline_experiment --push-to-hub <your-hf-username>/airline-rl-tuned
```
Then set `HF_MODEL_ID` in Cells 3 and 5 to `<your-hf-username>/airline-rl-tuned`.

### Why not use the HuggingFace Inference API directly?
HF's Inference Providers API (used by litellm's `huggingface/` prefix) only serves
models from approved providers (Groq, Together AI, etc.) — it cannot serve arbitrary
user-uploaded fine-tuned checkpoints on the free tier. vllm avoids this by serving
the model locally from the Colab GPU.